# PyTorch Fundamentals for AI Engineers

**Track:** Model Development  
**Prerequisites:** Notebook 13 (ML Fundamentals)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## Why This Notebook Exists

Notebook 13 taught you WHAT a model is. Notebook 15 will teach you LoRA fine-tuning with HuggingFace. But HuggingFace is a HIGH-LEVEL wrapper. When something breaks — and it WILL break — you need to understand the engine underneath.

**PyTorch IS that engine.** Every HuggingFace model, every LoRA adapter, every training loop in this curriculum runs on PyTorch.

```
NB13: What IS a model?       (concepts)
THIS: How do I BUILD one?    (PyTorch)
NB15: How do I FINE-TUNE?    (HuggingFace + PEFT)
```

---

## What You Will Learn

| Skill | Where Used in Curriculum |
|-------|------------------------|
| Tensors & GPU | Every model notebook (NB15, 18, 19) |
| Autograd (`.backward()`) | Training loops, Knowledge Distillation (NB15) |
| `nn.Module` | Every custom model, LoRA adapters (NB15) |
| `DataLoader` | Fine-tuning datasets (NB15, 12) |
| Custom training loop | Debugging HuggingFace Trainer issues |

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors call `model.fit()` and pray. Seniors understand that PyTorch's autograd graph, memory management, and data pipeline ARE the performance bottleneck in 90% of training issues. Debugging LoRA OOM errors requires understanding tensor dtypes and gradient accumulation at the PyTorch level.

**Mental Model:** Think of PyTorch as a calculator that remembers every operation you did. When you ask "what went wrong?" (`.backward()`), it replays the tape in reverse and tells you exactly how to fix each number.

**Common Junior Pitfall:** Moving tensors between CPU and GPU carelessly, causing silent slowdowns. One `.cpu()` call inside a training loop can make training 100x slower because data gets copied back and forth every step.

---

In [ ]:
# Cell 1 — Install
!pip install -q torch numpy matplotlib scikit-learn

## 1 · Tensors: The Foundation of Everything

A **tensor** is a multi-dimensional array — like a NumPy array but with two superpowers:
1. **GPU acceleration** — runs on NVIDIA GPUs for 10-100x speedups
2. **Automatic differentiation** — tracks gradients for training

| NumPy | PyTorch | Shape Example |
|-------|---------|---------------|
| `np.array([1,2,3])` | `torch.tensor([1,2,3])` | 1D: vector |
| `np.zeros((3,4))` | `torch.zeros(3,4)` | 2D: matrix |
| `np.random.randn(2,3,4)` | `torch.randn(2,3,4)` | 3D: batch of matrices |

### Data Types (dtypes) — This WILL Bite You

| dtype | Bits | Memory (1B params) | When to Use |
|-------|------|--------------------|---------|
| `float32` | 32 | 4 GB | Default, training |
| `float16` | 16 | 2 GB | Mixed-precision training |
| `bfloat16` | 16 | 2 GB | Modern GPU training (A100+) |
| `int8` | 8 | 1 GB | Quantized inference |

In [ ]:
# Cell 2 — Tensor basics
import torch
import numpy as np

# Creating tensors
a = torch.tensor([1.0, 2.0, 3.0])           # From Python list
b = torch.zeros(3, 4)                         # 3x4 matrix of zeros
c = torch.randn(2, 3)                         # Random normal (used for weight init)
d = torch.from_numpy(np.array([4.0, 5.0]))    # From NumPy

print(f'Vector: {a} | shape: {a.shape} | dtype: {a.dtype}')
print(f'Matrix: shape {b.shape}')
print(f'Random: {c}')

# dtype matters!
fp32 = torch.randn(1000, 1000)               # 4 MB
fp16 = fp32.half()                              # 2 MB (saves half the memory)
bf16 = fp32.bfloat16()                          # 2 MB (better for training)
print(f'\nfp32: {fp32.element_size()} bytes/element = {fp32.nelement()*fp32.element_size()/1e6:.1f} MB')
print(f'fp16: {fp16.element_size()} bytes/element = {fp16.nelement()*fp16.element_size()/1e6:.1f} MB')

# GPU check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
if device == 'cuda':
    gpu_tensor = a.to('cuda')  # Move tensor to GPU
    print(f'GPU tensor device: {gpu_tensor.device}')

In [ ]:
# Cell 3 — Tensor operations (the building blocks of neural networks)
import torch

# Every neural network is just these operations combined:
x = torch.randn(4, 3)   # Batch of 4 samples, 3 features each
W = torch.randn(3, 5)   # Weight matrix: 3 inputs -> 5 outputs
b = torch.randn(5)      # Bias vector

# Linear layer: y = x @ W + b (THIS is what nn.Linear does)
y = x @ W + b          # @ is matrix multiplication
print(f'Input:  {x.shape}  (batch=4, features=3)')
print(f'Weight: {W.shape}  (in=3, out=5)')
print(f'Output: {y.shape}  (batch=4, features=5)')

# Activation function (ReLU: replace negatives with 0)
activated = torch.relu(y)
print(f'\nBefore ReLU: {y[0,:3].tolist()}')
print(f'After ReLU:  {activated[0,:3].tolist()}  (negatives become 0)')

# Softmax (convert raw scores to probabilities)
logits = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(logits, dim=0)
print(f'\nLogits: {logits.tolist()} -> Probabilities: {[f"{p:.2%}" for p in probs.tolist()]}')
print(f'Sum of probabilities: {probs.sum():.4f} (always 1.0)')

---
## 2 · Autograd: The Magic Behind Training

### The Problem Autograd Solves

In NB13, you manually computed gradients: `dw = np.mean(2 * (y_pred - y_true) * X)`

For a model with 2.7 BILLION parameters (like Phi-2 in NB15), writing gradients by hand is impossible. **Autograd computes ALL gradients automatically.**

### How It Works

```
1. Set requires_grad=True on parameters you want to train
2. Do math (forward pass) — PyTorch records every operation
3. Call .backward() — PyTorch replays the tape in reverse
4. Read .grad on each parameter — that's the gradient!
```

In [ ]:
# Cell 4 — Autograd in action
import torch

# Simple example: f(x) = x^2 + 3x + 1, find df/dx at x=2
# Analytically: df/dx = 2x + 3 = 2(2) + 3 = 7

x = torch.tensor(2.0, requires_grad=True)  # Tell PyTorch to track this

# Forward pass (PyTorch records every operation)
f = x**2 + 3*x + 1
print(f'f(2) = {f.item():.1f}')   # 4 + 6 + 1 = 11

# Backward pass (PyTorch computes gradients automatically)
f.backward()

print(f'df/dx at x=2: {x.grad.item():.1f}')  # Should be 7.0
print(f'Verified: 2(2) + 3 = {2*2 + 3}')

# Now with a neural network weight update
print('\n--- Weight Update Demo ---')
w = torch.tensor(0.5, requires_grad=True)   # A model weight
x_data = torch.tensor(3.0)                   # Input
y_true = torch.tensor(6.0)                   # Target (true answer)

# Forward: predict
y_pred = w * x_data           # prediction = 0.5 * 3 = 1.5
loss = (y_pred - y_true)**2   # MSE loss = (1.5 - 6)^2 = 20.25

# Backward: compute gradient
loss.backward()
print(f'Prediction: {y_pred.item():.1f}, Loss: {loss.item():.2f}')
print(f'Gradient dL/dw: {w.grad.item():.1f}')

# Update weight (gradient descent step)
lr = 0.01
with torch.no_grad():  # Don't track this operation
    w_new = w - lr * w.grad
print(f'Weight: {w.item():.3f} -> {w_new.item():.3f} (moved toward correct answer)')

### 🚨 Autograd Gotchas (Will Save You Hours of Debugging)

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Forgetting `requires_grad=True` | `.grad` is `None` | Set on parameters or use `nn.Parameter` |
| Not calling `optimizer.zero_grad()` | Gradients accumulate across steps | Always zero before `.backward()` |
| Modifying tensors in-place during grad | `RuntimeError: in-place operation` | Use `with torch.no_grad():` for updates |
| Calling `.item()` inside training loop | Kills GPU performance | Only use for logging, not computation |

---

## 3 · `nn.Module`: Building Neural Networks

Every model in PyTorch — from a 2-layer MLP to GPT-4 — inherits from `nn.Module`. It gives you:
- Automatic parameter tracking (knows which tensors are weights)
- `.to('cuda')` moves ALL weights to GPU in one call
- Save/load with `torch.save()` / `torch.load()`

### The Two Methods You Must Define

```python
class MyModel(nn.Module):
    def __init__(self):          # Define layers
        super().__init__()
        self.layer1 = nn.Linear(10, 5)
    
    def forward(self, x):        # Define computation
        return self.layer1(x)
```

In [ ]:
# Cell 5 — Building a real neural network
import torch
import torch.nn as nn

class FraudDetector(nn.Module):
    """A 3-layer neural network for fraud detection.
    
    Architecture:
        Input (20 features) -> Hidden1 (64) -> Hidden2 (32) -> Output (1)
        
    This is the SAME architecture that HuggingFace wraps.
    When you call model(x) in NB15, THIS is what runs underneath.
    """
    
    def __init__(self, input_dim=20):
        super().__init__()  # MUST call this
        
        # Define layers (nn.Linear = y = Wx + b)
        self.layer1 = nn.Linear(input_dim, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        
        # Activation functions
        self.relu = nn.ReLU()          # Removes negatives
        self.dropout = nn.Dropout(0.2) # Randomly zeros 20% of neurons (prevents overfitting)
        self.sigmoid = nn.Sigmoid()    # Squashes output to 0-1 (probability)
    
    def forward(self, x):
        """Forward pass: input -> prediction."""
        x = self.relu(self.layer1(x))   # Layer 1 + activation
        x = self.dropout(x)             # Regularization
        x = self.relu(self.layer2(x))   # Layer 2 + activation
        x = self.sigmoid(self.output(x)) # Output probability
        return x

# Create model
model = FraudDetector(input_dim=20)

# Inspect it
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: FraudDetector')
print(f'  Total parameters: {total_params:,}')
print(f'  Trainable: {trainable:,}')
print(f'\nArchitecture:')
print(model)

# Test with fake data
fake_batch = torch.randn(8, 20)  # 8 transactions, 20 features each
predictions = model(fake_batch)   # Forward pass
print(f'\nInput shape:  {fake_batch.shape}  (8 transactions, 20 features)')
print(f'Output shape: {predictions.shape}  (8 fraud probabilities)')
print(f'Predictions:  {[f"{p.item():.3f}" for p in predictions.squeeze()]}')

---
## 4 · DataLoader: Feeding Data Efficiently

You can't load 1 million training examples into GPU memory at once. `DataLoader` solves this by:
1. Splitting data into **batches** (e.g., 32 samples at a time)
2. **Shuffling** each epoch (prevents learning order-dependent patterns)
3. Loading data in **parallel** (multiple CPU workers prepare batches while GPU trains)

```
Dataset: [sample1, sample2, ..., sample1000]  (too big for GPU)
                    |
DataLoader(batch_size=32)
                    |
Epoch 1: [batch1(32)] -> [batch2(32)] -> ... -> [batch32(8)]
Epoch 2: [shuffled batch1] -> [shuffled batch2] -> ...
```

In [ ]:
# Cell 6 — Dataset and DataLoader
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class FraudDataset(Dataset):
    """Custom dataset for fraud detection.
    
    Every PyTorch dataset needs __len__ and __getitem__.
    HuggingFace datasets (NB15) follow this same pattern.
    """
    
    def __init__(self, num_samples=1000, num_features=20):
        np.random.seed(42)
        self.X = torch.randn(num_samples, num_features)
        # 5% fraud rate (imbalanced, like real-world)
        self.y = torch.zeros(num_samples, 1)
        self.y[:int(num_samples * 0.05)] = 1.0
        # Shuffle
        perm = torch.randperm(num_samples)
        self.X = self.X[perm]
        self.y = self.y[perm]
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = FraudDataset(800)
test_dataset = FraudDataset(200)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches (batch_size=32)')
print(f'Test:  {len(test_dataset)} samples, {len(test_loader)} batches (batch_size=64)')

# Peek at one batch
X_batch, y_batch = next(iter(train_loader))
print(f'\nOne batch: X={X_batch.shape}, y={y_batch.shape}')
print(f'Fraud in this batch: {y_batch.sum().int().item()} / {len(y_batch)}')

---
## 5 · The Complete Training Loop

This is the **most important section** in this notebook. Every training framework (HuggingFace Trainer, Lightning, TRL) is a wrapper around this exact loop:

```python
for epoch in range(num_epochs):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()          # 1. Clear old gradients
        predictions = model(batch_X)   # 2. Forward pass
        loss = loss_fn(predictions, batch_y)  # 3. Compute loss
        loss.backward()                # 4. Compute gradients
        optimizer.step()               # 5. Update weights
```

### Optimizers: Which One to Use?

| Optimizer | When to Use | Used In Curriculum |
|-----------|-----------|---|
| `SGD` | Simple models, learning | Rarely in production |
| `Adam` | Default for most tasks | Most training |
| `AdamW` | When using weight decay | NB15 (LoRA fine-tuning) |
| `Paged AdamW 8-bit` | Large models, limited GPU | NB15 (QLoRA) |

In [ ]:
# Cell 7 — Complete training loop
import torch
import torch.nn as nn

# 1. Model
model = FraudDetector(input_dim=20)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)  # Move model to GPU (if available)

# 2. Loss function (Binary Cross-Entropy for binary classification)
loss_fn = nn.BCELoss()

# 3. Optimizer (AdamW — same as used in LoRA fine-tuning in NB15)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# 4. Training loop
num_epochs = 10
print('Training FraudDetector...')
print(f'  Device: {device}')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')

for epoch in range(num_epochs):
    model.train()  # Enable dropout
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # THE 5 SACRED STEPS (memorize these)
        optimizer.zero_grad()                # 1. Clear old gradients
        predictions = model(batch_X)         # 2. Forward pass
        loss = loss_fn(predictions, batch_y) # 3. Compute loss
        loss.backward()                      # 4. Backward pass (compute gradients)
        optimizer.step()                     # 5. Update weights
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    # Evaluate
    model.eval()  # Disable dropout
    correct = 0
    total = 0
    with torch.no_grad():  # Don't compute gradients during evaluation
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            preds = (model(batch_X) > 0.5).float()
            correct += (preds == batch_y).sum().item()
            total += len(batch_y)
    
    accuracy = correct / total
    if epoch % 2 == 0 or epoch == num_epochs - 1:
        print(f'  Epoch {epoch+1:2d}: loss={avg_loss:.4f}, accuracy={accuracy:.2%}')

print(f'\nTraining complete!')

In [ ]:
# Cell 8 — Save and load models
import torch

# Save model weights (this is what HuggingFace .save_pretrained() does internally)
torch.save(model.state_dict(), 'fraud_detector.pt')
print(f'Model saved: fraud_detector.pt')
print(f'  File contains: {len(model.state_dict())} tensors')
for name, param in model.state_dict().items():
    print(f'    {name}: {param.shape}')

# Load model weights
new_model = FraudDetector(input_dim=20)
new_model.load_state_dict(torch.load('fraud_detector.pt', weights_only=True))
new_model.eval()
print(f'\nModel loaded successfully!')

# Verify same predictions
test_input = torch.randn(1, 20)
with torch.no_grad():
    p1 = model.cpu()(test_input).item()
    p2 = new_model(test_input).item()
print(f'Original model prediction: {p1:.6f}')
print(f'Loaded model prediction:   {p2:.6f}')
print(f'Identical: {abs(p1-p2) < 1e-6}')

---
## ✅ Knowledge Check

### Question 1: Autograd
Why must you call `optimizer.zero_grad()` at the start of each training step?

<details>
<summary>👉 Click to reveal answer</summary>

PyTorch **accumulates** gradients by default (adds new gradients to existing ones). Without zeroing, gradients from step 1 would still be there at step 2, causing incorrect updates. This accumulation feature exists intentionally for **gradient accumulation** (NB15) where you simulate larger batch sizes by accumulating over multiple mini-batches before stepping.
</details>

### Question 2: model.train() vs model.eval()
What changes when you switch from `model.train()` to `model.eval()`?

<details>
<summary>👉 Click to reveal answer</summary>

Two things change: (1) **Dropout** layers stop dropping neurons (all neurons are used for inference). (2) **BatchNorm** layers use running statistics instead of batch statistics. Forgetting `model.eval()` during inference is a common bug that causes inconsistent predictions.
</details>

### Question 3: `torch.no_grad()`
Why do we wrap evaluation in `with torch.no_grad():`?

<details>
<summary>👉 Click to reveal answer</summary>

During evaluation/inference, we don't need gradients (we're not training). `torch.no_grad()` tells PyTorch to **skip building the computational graph**, saving significant GPU memory and compute. Without it, evaluating a large model can OOM because PyTorch stores intermediate values for a backward pass that will never happen.
</details>

### Question 4: Device Mismatch
What happens if your model is on GPU but your input data is on CPU?

<details>
<summary>👉 Click to reveal answer</summary>

PyTorch raises a `RuntimeError: Expected all tensors to be on the same device`. You must ensure both model and data are on the same device. The fix: `data = data.to(device)` where `device` matches your model. This is the #1 most common PyTorch error for beginners.
</details>

### Question 5: dtype
In NB15, the training uses `bf16=True`. Why bfloat16 instead of float16?

<details>
<summary>👉 Click to reveal answer</summary>

bfloat16 has the same **exponent range** as float32 (8 bits) but fewer mantissa bits (7 vs 23). This means it can represent the same range of very large and very small numbers as float32 without overflow/underflow. float16 has a smaller exponent range (5 bits) and can cause training instability (loss becomes NaN). bfloat16 is the standard for modern GPU training on A100/H100.
</details>

---
## 🔨 Practical Practice

### Exercise 1: Build a Sentiment Classifier
Modify the `FraudDetector` class to create a `SentimentClassifier` with:
- Input: 768 features (BERT embedding size)
- Hidden layers: 256 -> 128 -> 64
- Output: 3 classes (positive, neutral, negative) using `nn.Softmax`
- Add `nn.BatchNorm1d` after each hidden layer

### Exercise 2: Learning Rate Experiment
Using the training loop from Cell 7, train the FraudDetector with three different learning rates: 0.1, 0.001, 0.00001. Plot all three loss curves. Which converges best?

### Exercise 3: GPU Memory Estimation
A model has 7 billion parameters in float32. Calculate:
1. How much GPU memory for the model weights?
2. How much for float16?
3. How much for 4-bit quantization?
4. Why does training require ~3x more memory than inference? (Hint: gradients + optimizer states)

In [ ]:
# ===== EXERCISE SOLUTIONS =====
import torch
import torch.nn as nn

# --- Exercise 1: Sentiment Classifier ---
class SentimentClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 3),
        )
    
    def forward(self, x):
        return self.net(x)  # Use CrossEntropyLoss (includes softmax)

clf = SentimentClassifier()
print('Ex 1: SentimentClassifier')
print(f'  Parameters: {sum(p.numel() for p in clf.parameters()):,}')
out = clf(torch.randn(4, 768))
print(f'  Output shape: {out.shape} (batch=4, classes=3)')

# --- Exercise 3: GPU Memory ---
print('\nEx 3: GPU Memory for 7B model')
params = 7e9
for name, bits in [('float32', 32), ('float16', 16), ('4-bit', 4)]:
    gb = params * bits / 8 / 1e9
    print(f'  {name}: {gb:.1f} GB')
print(f'  Training at fp16: ~{params*16/8/1e9*3:.0f} GB (weights + gradients + optimizer)')
print(f'  That is why QLoRA (NB15) uses 4-bit base + 16-bit adapters!')

---
## Summary

| Concept | What You Learned | Where It Appears Next |
|---------|-----------------|----------------------|
| Tensors + dtypes | GPU arrays with type control | NB15 (bfloat16 training), NB29 (quantization) |
| Autograd (`.backward()`) | Automatic gradient computation | NB15 (LoRA training), Knowledge Distillation |
| `nn.Module` | How to build models | NB15 (PEFT wraps your model with adapters) |
| `DataLoader` | Efficient batched data feeding | NB15 (SFTTrainer uses this internally) |
| Training loop | The 5 sacred steps | NB15 (HuggingFace Trainer wraps this) |
| Save/Load | `state_dict()` persistence | NB15 (model registry, adapter saving) |

**Next →** `15_model_development.ipynb` — LoRA Fine-Tuning with HuggingFace (now you'll understand what's happening underneath)